In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from torch.distributions import Normal

# ==========================================
# 1. CUSTOM GYMNASIUM ENVIRONMENT (H-RARL)
# ==========================================
class F18_HRARL_Env(gym.Env):
    """Custom 6-DOF F-18 Environment with Hierarchical Adversarial constraints."""
    def __init__(self):
        super(F18_HRARL_Env, self).__init__()

        # Low-Level State: 12-DOF Kinematics + 3-DOF Target Goal = 15
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(15,), dtype=np.float32)

        # Low-Level Action: Elevator, Aileron, Rudder, Throttle [-1, 1]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(4,), dtype=np.float32)

        # Internal State Variables
        self.kinematic_state = np.zeros(12) # [X, Y, Z, u, v, w, phi, theta, psi, p, q, r]
        self.current_goal = np.zeros(3)     # [Target Roll, Target G, Target Throttle]

        # Tactical State Variables
        self.threat_pos = np.array([1000.0, 1000.0, 5000.0])
        self.threat_velocity = np.array([-200.0, -200.0, 0.0])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.kinematic_state = np.random.uniform(-0.1, 0.1, 12)
        self.kinematic_state[2] = 10000.0 # Start at 10,000 ft
        self.current_goal = np.zeros(3)
        self.threat_pos = np.array([1000.0, 1000.0, 5000.0])
        return self._get_low_level_obs(), {}

    def _get_low_level_obs(self):
        return np.concatenate((self.kinematic_state, self.current_goal)).astype(np.float32)

    def get_high_level_obs(self):
        """Tactical array: 3D pos, energy, threat pos, closure rate"""
        pos = self.kinematic_state[0:3]
        energy = np.linalg.norm(self.kinematic_state[3:6]) + self.kinematic_state[2] # Simplified energy state
        closure_rate = np.linalg.norm(self.kinematic_state[3:6] - self.threat_velocity)
        return np.concatenate((pos, [energy], self.threat_pos, [closure_rate])).astype(np.float32)

    def set_high_level_goal(self, goal):
        """Called by the Meta-Controller every N steps"""
        self.current_goal = np.clip(goal, -1.0, 1.0) # Normalized goal parameters

        # Tactical Adversary: Unpredictably alter threat heading/speed
        adv_tactical_noise = np.random.uniform(-50, 50, 3)
        self.threat_velocity += adv_tactical_noise
        self.threat_pos += self.threat_velocity * 0.1

    def _update_6dof_physics(self, actions):
        """
        PLACEHOLDER: Replace this with your JSBSim / X-Plane API calls.
        Updates self.kinematic_state based on the 4 continuous actions.
        """
        # Simulated physics update for demonstration
        self.kinematic_state[6:9] += actions[0:3] * 0.1 # Update Euler angles based on stick
        self.kinematic_state[3] += actions[3] * 10.0    # Update speed based on throttle
        self.kinematic_state[2] -= 5.0                  # Gravity / drag bleed

    def step(self, action):
        """Low-level muscle-memory tracking step"""
        # Kinematic Adversary: Inject bounded noise (Wind shear/faults)
        adv_kinematic_noise = np.random.uniform(-0.05, 0.05, 4)
        noisy_action = np.clip(action + adv_kinematic_noise, -1.0, 1.0)

        # Step physics
        self._update_6dof_physics(noisy_action)

        # Low-Level Reward: Minimize error to High-Level goal
        current_roll, current_g, current_throttle = self.kinematic_state[6], self.kinematic_state[10], self.kinematic_state[3]
        error = np.linalg.norm(self.current_goal - np.array([current_roll, current_g, current_throttle]))
        reward = 1.0 - error

        # Terminal penalties (Altitude <= 0 or stall)
        terminated = False
        if self.kinematic_state[2] <= 0:
            reward -= 1000.0 # Massive crash penalty
            terminated = True

        return self._get_low_level_obs(), reward, terminated, False, {}

    def calculate_high_level_reward(self):
        """Calculated after the macro-step sequence completes"""
        dist_to_threat = np.linalg.norm(self.kinematic_state[0:3] - self.threat_pos)
        if dist_to_threat < 100:
            return -1000.0 # Intercepted
        return dist_to_threat * 0.1 # Reward maximizing distance

# ==========================================
# 2. NETWORKS (LSTM Low-Level & MLP High-Level)
# ==========================================
class LowLevelLSTM_PPO(nn.Module):
    """Goal-Conditioned Low-Level Policy with LSTM for temporal tracking."""
    def __init__(self, obs_dim=15, act_dim=4, hidden_size=64):
        super(LowLevelLSTM_PPO, self).__init__()
        self.hidden_size = hidden_size

        # LSTM feature extractor
        self.lstm = nn.LSTM(obs_dim, hidden_size, batch_first=True)

        # Actor Head (Muscle Memory)
        self.actor_mean = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),
            nn.Linear(64, act_dim),
            nn.Tanh() # Bounded [-1, 1] constraints
        )
        self.actor_log_std = nn.Parameter(torch.zeros(1, act_dim))

        # Critic Head (Value Estimation)
        self.critic = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x, hidden_state):
        # x shape: (batch, seq_len, obs_dim)
        lstm_out, hidden_state = self.lstm(x, hidden_state)
        lstm_out = lstm_out[:, -1, :] # Take last output of sequence

        action_mean = self.actor_mean(lstm_out)
        action_std = self.actor_log_std.exp().expand_as(action_mean)
        dist = Normal(action_mean, action_std)

        value = self.critic(lstm_out)
        return dist, value, hidden_state

class HighLevelPPO(nn.Module):
    """PPO Meta-Controller (The Tactician) generating parameterized goals."""
    def __init__(self, obs_dim=8, goal_dim=3):
        super(HighLevelPPO, self).__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, goal_dim),
            nn.Tanh() # Output goals bounded [-1, 1]
        )
        self.actor_log_std = nn.Parameter(torch.zeros(1, goal_dim))

        self.critic = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        mean = self.actor(x)
        std = self.actor_log_std.exp().expand_as(mean)
        dist = Normal(mean, std)
        val = self.critic(x)
        return dist, val

# ==========================================
# 3. HIERARCHICAL TRAINING LOOP
# ==========================================
def train_hrarl():
    env = F18_HRARL_Env()

    # Initialize Networks
    low_level_agent = LowLevelLSTM_PPO()
    high_level_agent = HighLevelPPO()

    # Optimizers
    ll_optimizer = optim.Adam(low_level_agent.parameters(), lr=3e-4)
    hl_optimizer = optim.Adam(high_level_agent.parameters(), lr=1e-4)

    MACRO_STEPS = 10  # High-level steps every 10 low-level timesteps
    EPISODES = 1000

    for episode in range(EPISODES):
        ll_obs, _ = env.reset()
        hl_obs = env.get_high_level_obs()

        # Initialize LSTM hidden state (h_0, c_0)
        hidden = (torch.zeros(1, 1, 64), torch.zeros(1, 1, 64))

        done = False
        episode_ll_reward = 0
        episode_hl_reward = 0

        while not done:
            # ---------------------------------------------------------
            # 1. HIGH-LEVEL STEP (Tactician sets the goal)
            # ---------------------------------------------------------
            hl_obs_tensor = torch.FloatTensor(hl_obs).unsqueeze(0)
            hl_dist, hl_value = high_level_agent(hl_obs_tensor)
            goal_action = hl_dist.sample()

            env.set_high_level_goal(goal_action.squeeze(0).numpy())

            # ---------------------------------------------------------
            # 2. LOW-LEVEL TRACKING LOOP (Muscle Memory tracks the goal)
            # ---------------------------------------------------------
            for _ in range(MACRO_STEPS):
                # Format sequence for LSTM: (batch=1, seq_len=1, obs_dim=15)
                ll_obs_tensor = torch.FloatTensor(ll_obs).unsqueeze(0).unsqueeze(0)

                ll_dist, ll_value, hidden = low_level_agent(ll_obs_tensor, hidden)
                physical_action = ll_dist.sample()

                ll_obs_next, ll_reward, terminated, truncated, _ = env.step(physical_action.squeeze().numpy())

                episode_ll_reward += ll_reward
                ll_obs = ll_obs_next

                if terminated or truncated:
                    done = True
                    break

            # ---------------------------------------------------------
            # 3. HIGH-LEVEL REWARD & OBSERVATION UPDATE
            # ---------------------------------------------------------
            hl_reward = env.calculate_high_level_reward()
            episode_hl_reward += hl_reward
            hl_obs = env.get_high_level_obs()

            # Note: A full PPO implementation requires storing log_probs, values,
            # and rewards in a rollout buffer here, followed by Generalized Advantage
            # Estimation (GAE) and surrogate loss optimization.

        print(f"Episode {episode} | High-Level Tactical Reward: {episode_hl_reward:.2f} | Low-Level Tracking Reward: {episode_ll_reward:.2f}")

if __name__ == "__main__":
    # execute the training loop
    train_hrarl()

Episode 0 | High-Level Tactical Reward: 83665.18 | Low-Level Tracking Reward: -2773226.08
Episode 1 | High-Level Tactical Reward: 112873.07 | Low-Level Tracking Reward: -3114522.62
Episode 2 | High-Level Tactical Reward: 241419.58 | Low-Level Tracking Reward: -2143084.16
Episode 3 | High-Level Tactical Reward: 236596.77 | Low-Level Tracking Reward: -2592322.27
Episode 4 | High-Level Tactical Reward: 181914.41 | Low-Level Tracking Reward: -2338846.51
Episode 5 | High-Level Tactical Reward: 237513.92 | Low-Level Tracking Reward: -2931612.63
Episode 6 | High-Level Tactical Reward: 175201.40 | Low-Level Tracking Reward: -2747877.93
Episode 7 | High-Level Tactical Reward: 203558.24 | Low-Level Tracking Reward: -2961453.65
Episode 8 | High-Level Tactical Reward: 247583.62 | Low-Level Tracking Reward: -2984076.11
Episode 9 | High-Level Tactical Reward: 322788.77 | Low-Level Tracking Reward: -2401112.43
Episode 10 | High-Level Tactical Reward: 360281.34 | Low-Level Tracking Reward: -2517382.94

In [3]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import torch

def record_and_plot_trajectory(env, model, hidden_state, maneuver_name="F-18 Maneuver"):
    """
    Executes a deterministic rollout of the trained policy and plots the 3D flight path.
    """
    obs, _ = env.reset()

    # Lists to store positional data
    north_x = []
    east_y = []
    alt_z = []

    done = False
    step_count = 0
    max_steps = 1000  # Prevent infinite loops

    # 1. Deterministic Execution
    with torch.no_grad():
        while not done and step_count < max_steps:
            # Format sequence for LSTM: (batch=1, seq_len=1, obs_dim=15)
            obs_tensor = torch.FloatTensor(obs).unsqueeze(0).unsqueeze(0)

            # Extract only the mean from the Actor network (turn off exploratory noise)
            lstm_out, hidden_state = model.lstm(obs_tensor, hidden_state)
            lstm_out = lstm_out[:, -1, :]
            deterministic_action = model.actor_mean(lstm_out).squeeze().numpy()

            # Step the environment
            obs_next, _, terminated, truncated, _ = env.step(deterministic_action)

            # Record 3D coordinates (assuming state indices 0=X, 1=Y, 2=Z)
            north_x.append(obs[0])
            east_y.append(obs[1])
            alt_z.append(obs[2])

            obs = obs_next
            step_count += 1

            if terminated or truncated:
                done = True

    # 2. Coordinate Mapping & Plotting
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    # Plot the continuous flight path
    # Note: We swap standard axes so East is horizontal and North is vertical
    ax.plot(east_y, north_x, alt_z, label='Trajectory', color='blue', linewidth=2.5)

    # 3. Visual Cues
    # Green dot for entry, Red dot for exit
    ax.scatter(east_y[0], north_x[0], alt_z[0], color='green', s=100, label='Entry Point', zorder=5)
    ax.scatter(east_y[-1], north_x[-1], alt_z[-1], color='red', s=100, label='Exit Point', zorder=5)

    ax.set_xlabel('East (Position Y)')
    ax.set_ylabel('North (Position X)')
    ax.set_zlabel('Altitude (Position Z)')
    ax.set_title(f'3D Flight Path: {maneuver_name}')
    ax.legend()

    plt.show()